#An exercise in text classification: Guessing sentiment from movie criticism

We are going to work on a database of movie reviews. Each review has been annotated as positive (1) or negative (0).

To work in this case we will use a T4 type of virtual computer. Select it on the upper right side of colab, where it states "Connect", go to "Change Runtime type"


## 1. Download and import the dataset.

In [ ]:
 %pip install -q -U datasets

In [ ]:
from datasets import load_dataset
import pandas as pd

In [ ]:
data = load_dataset("rotten_tomatoes")
train_data = pd.DataFrame(data['train'])

## 2. Explore the data

# 3. We need to convert the text into date amenable to a machine learning algortithm.

For this we will use the TfidfVectorizer. Test how this works on the first 5 texts

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
train_data.iloc[:5].text

In [ ]:
vectorizer = TfidfVectorizer()
pd.DataFrame(
    vectorizer.fit_transform(train_data.iloc[:5].text).todense(),
    columns=vectorizer.get_feature_names_out()
)

## 4. Classify the reviews

Use the Logistic regression classifier and the TfidfVectorizer to classify the *reviews*.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn import linear_model
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import (
    LearningCurveDisplay, StratifiedKFold, train_test_split
)

## 4.1 Now classify the reviews reducing the dimensions or learning the manifold. Specifically use PCA and Kernel PCA for this. Only use few dimensions, a maximum of 20. Use the learning curve to choose the number of dimensions.



In [ ]:
from sklearn.decomposition import PCA, KernelPCA


## 5. Large language models !

Now we will use a large language model [bert](https://en.wikipedia.org/wiki/BERT_(language_model))
as a feature extractor

In [ ]:
from sentence_transformers import SentenceTransformer

# Load model
model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
train_embeddings = model.encode(data["train"]["text"], show_progress_bar=True)

Now that we have extracted the features to a new variable `train_embeddings` try the classifier again. Use these features as input for the classifier.


## Just as an example. LLM Prompting !
Now let's just ask an LLM for this, specifically [T5](https://en.wikipedia.org/wiki/T5_(language_model))
We will use the test data for convenience as it has only around 1000 records

In [ ]:
from transformers import pipeline as tpipeline
from transformers.pipelines.pt_utils import KeyDataset
import numpy as np
from tqdm import tqdm

In [ ]:
# Load our model
pipe = tpipeline(
    "text2text-generation",
    model="google/flan-t5-small",
    device="cuda:0"
)

In [ ]:
# Store the data in a dataframe

data_df = pd.DataFrame(data["test"])

In [ ]:
# Set up the prompt that we will ask to the T5 llm
prompt = "Is the following sentence positive or negative? "
llm_response = data.map(lambda example: {"t5": prompt + example['text']})


In [ ]:
# Run the prompt for every record and store the result

y_pred = []
for output in tqdm(pipe(KeyDataset(llm_response["test"], "t5"))):
    text = output[0]["generated_text"]
    y_pred.append(0 if text == "negative" else 1)

In [ ]:
# Convert the result to pandas
y_pred = pd.Series(y_pred)

In [ ]:
# Compute the accuracy
from sklearn.metrics import accuracy_score

print(f"Accuracy {accuracy_score(data_df.label, y_pred):.2f}")